In [2]:
%%sql

SHOW TABLES

Empty set


In [13]:
%%sql

DROP TABLE IF EXISTS data_gen;

OK


In [3]:
%%sql

CREATE TABLE data_gen (
    id STRING,
    name STRING,
    price DOUBLE
) WITH (
    'connector' = 'datagen',
    'rows-per-second' = '1',
    'fields.price.max' = '100',
    'fields.price.min' = '1'
);

OK


In [4]:
%%sql

CREATE TABLE gen_data (
    id STRING,
    name STRING,
    price DOUBLE
) WITH (
    'connector' = 'kafka',
    'topic' = 'gen_data',
    'properties.bootstrap.servers' = 'kafka:9092',
    'format' = 'json',
    'scan.startup.mode' = 'latest-offset'
);

OK


In [5]:
%%sql

INSERT INTO gen_data
SELECT * FROM data_gen;

Job submitted: 44cb88c070f862a46df5024b735ad572


In [19]:
%%sql

DROP TABLE proc_data

OK


In [6]:
%%sql

CREATE TABLE proc_data (
    id STRING,
    name STRING,
    price DOUBLE
) WITH (
    'connector' = 'kafka',
    'topic' = 'processed_data',
    'properties.bootstrap.servers' = 'kafka:9092',
    'value.format' = 'avro-confluent',
    'value.avro-confluent.url' = 'http://schema-registry:8081',
    'scan.startup.mode' = 'latest-offset'
);

OK


In [7]:
%%sql

INSERT INTO proc_data
SELECT 
    id,
    'XD' || name,
    price * 10
FROM gen_data;

Job submitted: 3858fa93d3efaba8ddc7ea90b102e25c
